# reduce-gather-sum — worked example 2: reduce(SUM) leaves only dst with the total

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reduce-gather-sum`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`dist.reduce(tensor, dst, op=SUM)` aggregates across all ranks but writes the result ONLY to the destination rank; non-dst ranks keep an unspecified value. This contrasts with `all_reduce`, where every rank ends up with the total. Knowing where the result lands matters when only one rank logs or checkpoints.

## Worked solution

Our `FakeDist` simulates `reduce` by computing the true sum of all ranks' contributions but only writing it into the destination rank's tensor; for other ranks it leaves the original value untouched (here we mimic that by writing back the rank's own contribution). The function `reduce_sum_to_dst` starts each rank with `tensor = [rank+1]`, calls `reduce(..., dst=0, SUM)`, and reports the post-reduce value. On rank 0 we get the full sum; on other ranks we get their own value, demonstrating that the aggregate is dst-only. We print both the dst result and a non-dst result to make the asymmetry visible.

In [ ]:
class FakeDist:
    def __init__(self, contribs):
        self.contribs = list(contribs)  # one scalar per rank
        self.total = float(sum(self.contribs))
    def reduce(self, tensor, dst, rank):
        if rank == dst:
            tensor.copy_(t.tensor([self.total]))
        # non-dst ranks keep their original value (unspecified in real torch)


def reduce_sum_to_dst(rank, dist_module):
    tensor = t.tensor([float(rank + 1)])
    dist_module.reduce(tensor, dst=0, rank=rank)
    return tensor.item()


contribs = [1.0, 2.0, 3.0]
fd = FakeDist(contribs)
print('rank 0 (dst):', reduce_sum_to_dst(0, fd))
print('rank 2 (non-dst):', reduce_sum_to_dst(2, fd))